# RAG sur de la documentation technique francophone (MDN FR)
**Cours : Large Language Models — projet pratique**

Ce notebook fait tourner de bout en bout un système **RAG (Retrieval-Augmented
Generation)** sur la documentation web de MDN en français, et compare :
- **closed-book** (le LLM seul) vs **RAG** (LLM + récupération) ;
- l'effet du nombre de passages **k** ;
- un retrieveur **de base** vs **spécialisé** (fine-tuné).

> ⚠️ Avant de lancer : *Exécution → Modifier le type d'exécution → GPU T4*.


## 0. Vérifier le GPU

In [ ]:
!nvidia-smi

## 1. Récupérer le code et installer les dépendances
Remplacez l'URL par celle de **votre** dépôt après l'avoir poussé sur GitHub.

In [ ]:
# Si vous travaillez depuis le dépôt cloné :
!git clone https://github.com/VOTRE_USER/rag-doc-technique-fr.git
%cd rag-doc-technique-fr
!pip install -q -r requirements.txt

## 2. Construire le corpus (clone partiel de MDN + nettoyage + découpage)

In [ ]:
!python scripts/01_build_corpus.py

Aperçu de quelques passages :

In [ ]:
import json
chunks = [json.loads(l) for l in open('data/chunks.jsonl', encoding='utf-8')]
print('Nombre de passages :', len(chunks))
print(chunks[0]['title'])
print(chunks[0]['text'][:300], '…')

## 3. Indexer les passages (embeddings + FAISS)

In [ ]:
!python scripts/02_build_index.py

## 4. Démo interactive du RAG
On charge le retrieveur + le générateur et on pose une question.

In [ ]:
import sys; sys.path.insert(0, '.')
from ragdoc.retriever import Retriever
from ragdoc.generator import Generator
from ragdoc.pipeline import RagPipeline

retriever = Retriever.load()
generator = Generator()
rag = RagPipeline(retriever, generator)

out = rag.answer("À quoi sert l'élément HTML <article> ?")
print("RÉPONSE :", out['answer'])
print("\nSOURCES :")
for s in out['sources']:
    print(' -', s['title'], '|', round(s['score'], 3))

## 5. Générer le jeu d'évaluation
On crée des paires (question, réponse, passage source) à partir du corpus.
*La génération est lente : commencez avec un petit `-n` pour tester.*
Le fichier produit (`data/eval_set.jsonl`) est ensuite versionné dans le dépôt :
c'est lui qui rend vos résultats reproductibles.

In [ ]:
!python scripts/03_make_eval_set.py -n 100

## 6. Évaluation : RAG vs closed-book + ablation sur k
Produit `results/report_base.json`.

In [ ]:
!python scripts/04_run_evaluation.py --n_gen 30

## 7. (Optionnel) Spécialiser le retrieveur, puis comparer
On fine-tune l'embedding model sur le domaine, on réindexe, et on réévalue
pour mesurer le gain de la spécialisation.

In [ ]:
!python scripts/05_finetune_embedder.py --epochs 2
!python scripts/02_build_index.py --finetuned
!python scripts/04_run_evaluation.py --finetuned --n_gen 30

## 8. Comparer les rapports
Tableau récapitulatif base vs spécialisé.

In [ ]:
import json, pandas as pd
rows = []
for tag in ['base', 'finetuned']:
    try:
        r = json.load(open(f'results/report_{tag}.json', encoding='utf-8'))
    except FileNotFoundError:
        continue
    d = {'index': tag, **r['retrieval']}
    if 'generation' in r:
        d['F1_RAG'] = r['generation']['rag']['F1']
        d['F1_closed_book'] = r['generation']['closed_book']['F1']
    rows.append(d)
pd.DataFrame(rows)

## 9. Analyse et conclusions
À remplir avec **vos** résultats :
- Le RAG améliore-t-il F1/EM par rapport au closed-book ? De combien ?
- Comment évolue Hit@k quand k augmente ? Y a-t-il un palier ?
- La spécialisation du retrieveur améliore-t-elle Hit@k / MRR ?
- Limites observées (qualité du petit LLM, bruit du corpus, etc.).
